In [1]:
import sys
sys.path.append("../src")

import os
from typing import Dict, Any

import torch
import numpy as np
import pandas as pd
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from utils import *

In [2]:
data_path = os.path.join("..", "data", "raw", "gene-expression-normalized.csv")

df = pd.read_csv(data_path, index_col=0)

df.head()

,A1BG,A1CF,A2M,A2ML1,A3GALT2,A4GALT,A4GNT,AAAS,AACS,AADAC,...,ZWILCH,ZWINT,ZXDA,ZXDB,ZXDC,ZYG11A,ZYG11B,ZYX,ZZEF1,ZZZ3
ACH-000828,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
ACH-000568,0.0,0.122203,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
ACH-000560,0.0,0.152391,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
ACH-000561,0.0,0.160657,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
ACH-000562,0.0,0.161598,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [3]:
df.loc["ACH-000828", :]

A1BG       0.0
A1CF       0.0
A2M        0.0
A2ML1      0.0
A3GALT2    0.0
          ... 
ZYG11A     0.0
ZYG11B     0.0
ZYX        0.0
ZZEF1      0.0
ZZZ3       0.0
Name: ACH-000828, Length: 19264, dtype: float64

In [4]:
pretrain_gene_x = torch.tensor(df.iloc[0, :].values).unsqueeze(0)

data_gene_ids = torch.arange(19264).repeat(pretrain_gene_x.shape[0], 1)

In [5]:
class scFoundationDataset(Dataset):
    def __init__(
        self,
        gene_expression_path: str, # normalized
        pretrain_config: Dict[str, Any]
        ):
        super().__init__()

        self.gene_df = pd.read_csv(gene_expression_path, index_col=0)
        self.pretrain_config = pretrain_config

    def __len__(self):
        return len(self.gene_df)
    
    def __getitem__(self, idx):
        row = self.gene_df.iloc[idx, :].values

        if self.pretrain_config["rawcount"] == False:
            pretrain_gene_x = torch.tensor(row.unsqueeze(0))


        else:
            total_count = row.sum()
            pretrain_gene_x = torch.tensor(row.tolist() + [total_count, total_count]).unsqueeze(0)
        
        data_gene_ids = torch.arange(19_266).repeat(pretrain_gene_x.shape[0], 1)
        print("data gene ids local:", data_gene_ids.shape)
        value_labels = pretrain_gene_x > 0

        x, x_padding = gatherData(
            pretrain_gene_x, 
            value_labels, 
            self.pretrain_config["pad_token_id"]
            )
        
        position_gene_ids, _ = gatherData(
            data_gene_ids,
            value_labels,
            self.pretrain_config["pad_token_id"]
        )
    
        return x, x_padding, position_gene_ids

In [6]:
model_path = os.path.join("..", "assets", "models", "models.ckpt")

os.path.isfile(model_path)

True

In [7]:
model, config = load_model_frommmf(model_path)

/Users/ericmonzon/Desktop/personal projects/scFoundation-CDR/notebooks/../src/utils/load.py:126: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model_data = torch.load(best_c

{'mask_gene_name': False, 'gene_num': 19266, 'seq_len': 19266, 'encoder': {'hidden_dim': 768, 'depth': 12, 'heads': 12, 'dim_head': 64, 'seq_len': 19266, 'module_type': 'transformer', 'norm_first': False}, 'decoder': {'hidden_dim': 512, 'depth': 6, 'heads': 8, 'dim_head': 64, 'module_type': 'performer', 'seq_len': 19266, 'norm_first': False}, 'n_class': 104, 'pad_token_id': 103, 'mask_token_id': 102, 'bin_num': 100, 'bin_alpha': 1.0, 'rawcount': True, 'model': 'mae_autobin', 'test_valid_train_idx_dict': '/nfs_beijing/minsheng/data/os10000w-new/global_shuffle/meta.csv.train_set_idx_dict.pt', 'valid_data_path': '/nfs_beijing/minsheng/data/valid_count_10w.npz', 'num_tokens': 13, 'train_data_path': None, 'isPanA': False, 'isPlanA1': False, 'max_files_to_load': 5, 'bin_type': 'auto_bin', 'value_mask_prob': 0.3, 'zero_mask_prob': 0.03, 'replace_prob': 0.8, 'random_token_prob': 0.1, 'mask_ignore_token_ids': [0], 'decoder_add_zero': True, 'mae_encoder_max_seq_len': 15000, 'isPlanA': False, 'ma

In [8]:
dataset = scFoundationDataset(data_path, config)

In [9]:
x, x_padding, position_gene_ids = next(iter(dataset))

data gene ids local: torch.Size([1, 19266])


In [10]:
iter_dataset = iter(dataset)
_, _, _ = next(iter_dataset)

x, x_padding, position_gene_ids = next(iter_dataset)

data gene ids local: torch.Size([1, 19266])
data gene ids local: torch.Size([1, 19266])


In [11]:
x.shape

torch.Size([1, 665])

In [12]:
x_padding.shape

torch.Size([1, 665])

In [13]:
x_padding.shape

torch.Size([1, 665])

In [14]:
position_gene_ids.shape

torch.Size([1, 665])

In [15]:
position_gene_ids

tensor([[    1,    96,   101,   102,   151,   187,   190,   232,   235,   399,
           401,   402,   505,   525,   527,   528,   543,   572,   614,   668,
           828,   855,   902,   903,   944,   960,   968,   969,   971,   993,
           994,   995,  1056,  1156,  1163,  1164,  1177,  1205,  1209,  1228,
          1305,  1310,  1343,  1344,  1351,  1417,  1418,  1427,  1428,  1464,
          1465,  1466,  1467,  1472,  1477,  1478,  1480,  1483,  1484,  1485,
          1489,  1490,  1491,  1547,  1549,  1557,  1585,  1633,  1636,  1637,
          1641,  1643,  1656,  1698,  1702,  1716,  2072,  2129,  2152,  2156,
          2187,  2217,  2222,  2224,  2254,  2255,  2256,  2257,  2258,  2390,
          2474,  2477,  2478,  2479,  2480,  2482,  2508,  2511,  2560,  2562,
          2604,  2605,  2606,  2656,  2666,  2667,  2668,  2673,  2701,  2714,
          2721,  2731,  2732,  2734,  2738,  2756,  2776,  2863,  2977,  2980,
          2982,  2990,  2999,  3055,  3087,  3091,  

In [16]:
x.shape

torch.Size([1, 665])

In [17]:
torch.unsqueeze(x, 2).shape

torch.Size([1, 665, 1])

In [18]:
x = model.token_emb(torch.unsqueeze(x, 2).float(), output_weight=0)

x.shape

torch.Size([1, 665, 768])

In [19]:
x

tensor([[[ 0.1339, -0.0116,  0.1081,  ..., -0.1745,  0.0391,  0.1302],
         [ 0.4448,  0.1495,  0.0665,  ..., -0.2864,  0.7138,  0.2269],
         [ 0.4453,  0.0473,  0.1334,  ..., -0.3106,  0.8664,  0.4082],
         ...,
         [ 0.4459,  0.1277,  0.0824,  ..., -0.2922,  0.7528,  0.2671],
         [ 0.0072, -0.5034, -0.8472,  ..., -0.9954, -0.7295, -0.3049],
         [ 0.0072, -0.5034, -0.8472,  ..., -0.9954, -0.7295, -0.3049]]],
       grad_fn=<CopySlices>)

In [20]:
position_embedding = model.pos_emb(position_gene_ids)

position_embedding.shape

torch.Size([1, 665, 768])

In [21]:
x = x + position_embedding

x

tensor([[[ 1.1149,  0.1572,  1.0189,  ...,  0.6924,  0.0996, -0.3868],
         [ 2.3159,  0.0795, -0.9369,  ...,  0.4799,  0.2408,  2.5035],
         [-1.0266,  0.6073, -1.0404,  ..., -0.6142,  1.3649,  1.4919],
         ...,
         [ 0.0959, -0.3112, -0.1588,  ...,  2.1208,  0.5461, -1.8728],
         [-0.2786,  0.0434, -0.3019,  ...,  0.1752,  0.5093, -0.3669],
         [-0.6841, -2.3200, -0.2103,  ..., -1.4235, -0.8999,  0.3883]]],
       grad_fn=<AddBackward0>)

In [22]:
x_padding.any().item()

False

In [23]:
gene_embedding = model.encoder(x, x_padding)

In [24]:
gene_embedding.shape

torch.Size([1, 665, 768])

In [25]:
position_embedding

tensor([[[ 0.9809,  0.1689,  0.9108,  ...,  0.8669,  0.0605, -0.5170],
         [ 1.8711, -0.0700, -1.0034,  ...,  0.7663, -0.4731,  2.2766],
         [-1.4719,  0.5600, -1.1738,  ..., -0.3036,  0.4985,  1.0837],
         ...,
         [-0.3500, -0.4389, -0.2411,  ...,  2.4130, -0.2067, -2.1400],
         [-0.2858,  0.5468,  0.5452,  ...,  1.1705,  1.2388, -0.0620],
         [-0.6913, -1.8166,  0.6369,  ..., -0.4281, -0.1704,  0.6933]]],
       grad_fn=<EmbeddingBackward0>)

In [26]:
gene_embedding

tensor([[[ 0.8974,  1.1018, -0.2631,  ...,  1.1402, -0.6219,  0.2638],
         [-1.5361,  1.3223, -1.2432,  ..., -0.6372, -1.0033,  0.1299],
         [-0.8850,  0.5799, -0.9328,  ..., -0.0492, -1.6691, -0.1047],
         ...,
         [-1.7613,  0.8523, -1.0602,  ..., -0.3315, -1.2595, -0.7467],
         [-1.3340,  0.7855, -0.3573,  ...,  1.1677, -0.6078, -0.0445],
         [-1.5849,  1.0842, -1.9315,  ..., -0.4646, -1.2884, -0.1080]]],
       grad_fn=<NativeLayerNormBackward0>)

In [27]:
model.pos_emb(torch.tensor(1)).shape

torch.Size([768])

In [28]:
model.pos_emb

Embedding(19267, 768)

In [29]:
position_gene_ids.shape

torch.Size([1, 665])

In [30]:
df

,A1BG,A1CF,A2M,A2ML1,A3GALT2,A4GALT,A4GNT,AAAS,AACS,AADAC,...,ZWILCH,ZWINT,ZXDA,ZXDB,ZXDC,ZYG11A,ZYG11B,ZYX,ZZEF1,ZZZ3
ACH-000828,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
ACH-000568,0.0,0.122203,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
ACH-000560,0.0,0.152391,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
ACH-000561,0.0,0.160657,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
ACH-000562,0.0,0.161598,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
ACH-000242,0.0,0.108701,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
ACH-000245,0.0,0.112987,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
ACH-000244,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
ACH-000247,0.0,2.321917,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [31]:
gene_embedding.shape

torch.Size([1, 665, 768])

In [32]:
max_embed, _ = torch.max(gene_embedding, dim=1)

max_embed.shape

torch.Size([1, 768])

In [33]:
class scFoundationEncoder(nn.Module):
    def __init__(self, model: nn.Module):
        self.token_embedder = model.token_embed
        self.position_embedder = model.pos_embed
        self.encoder = model.encoder

    def forward(self, x, position_gene_ids, x_padding):
        x = self.token_embedder(torch.unsqueeze(x, 2).float(), output_weight=0)
        position_embedding = self.position_embedder(position_gene_ids)

        x = x + position_embedding
        gene_embedding = self.encoder(x, x_padding)

        gene_embedding, _ = torch.max(gene_embedding, dim=1)

        return gene_embedding

In [34]:
sample = torch.tensor(
    [[-1.2360, -0.2942, -0.1222,  0.8475],
    [ 1.1949, -1.1127, -2.2379, -0.6702],
    [ 1.5717, -0.9207,  0.1297, -1.8768],
    [-0.6172,  1.0036, -0.6060, -0.2432]]
    )

sample.shape

torch.Size([4, 4])

In [35]:
sample[0]

tensor([-1.2360, -0.2942, -0.1222,  0.8475])

In [36]:
drug_dict

NameError: name 'drug_dict' is not defined